In [ ]:
from reservoirpy.nodes import Reservoir
from sklearn.linear_model import Lasso, LassoCV
import numpy as np

# ----------------------------------------------------------
# 1. BUILD RESERVOIR (same as before, but standalone)
# ----------------------------------------------------------
reservoir = Reservoir(
    units=500,
    sr=0.95,
    lr=0.3,
    input_scaling=0.1,
    rc_connectivity=0.1,
    seed=42
)

# ----------------------------------------------------------
# 2. COLLECT RESERVOIR STATES
# ----------------------------------------------------------
warmup = 100

# Run reservoir on training data to get states
states_train = np.array(reservoir.run(X_train))  # shape (train_len, 500)

# Discard warmup states (reservoir hasn't washed in yet)
states_train = states_train[warmup:]
Y_train_cut  = Y_train[warmup:]

# ----------------------------------------------------------
# 3. FIT LASSO REGRESSION
# ----------------------------------------------------------
# Option A: Fixed alpha (you choose the regularization strength)
lasso = Lasso(alpha=1e-6, max_iter=10000, fit_intercept=True)
lasso.fit(states_train, Y_train_cut)

# Option B: Auto-select alpha by cross-validation (recommended)
# lasso = LassoCV(cv=5, max_iter=10000, fit_intercept=True)
# lasso.fit(states_train, Y_train_cut)
# print(f"Best alpha chosen by CV: {lasso.alpha_:.2e}")

# Check sparsity: how many readout weights are exactly zero?
n_zero = np.sum(lasso.coef_ == 0)
n_total = len(lasso.coef_)
print(f"Readout weights: {n_total} total, {n_zero} zero ({100*n_zero/n_total:.1f}% sparse)")

# ----------------------------------------------------------
# 4. CLOSED-LOOP PREDICTION
# ----------------------------------------------------------
# Washout: feed true test data to set reservoir state
_ = reservoir.run(X_test[:200])

# Free-run
pred_len = 500
Y_pred = []
current_input = X_test[200:201]

for _ in range(pred_len):
    # Get reservoir state for current input
    state = np.array(reservoir.run(current_input))  # shape (1, 500)
    # Apply LASSO readout manually
    pred = lasso.predict(state)                      # shape (1,)
    Y_pred.append(pred[0])
    current_input = pred.reshape(1, -1)

Y_pred = np.array(Y_pred)
Y_true = Y_test[200:200 + pred_len, 0]

rmse = np.sqrt(np.mean((Y_true - Y_pred) ** 2))
nrmse = rmse / np.std(Y_true)
print(f"RMSE:  {rmse:.6f}")
print(f"NRMSE: {nrmse:.6f}")